# Poster Figures
Re-renders key result plots with poster-ready styling.
No metrics are recomputed — all data loaded from existing CSVs.

**Style:** sans-serif, 300 DPI, 8×5 minimum, white background, light dotted y-grid.
**Palette:** Popularity=gray, CF=blue, Content=green, Hybrid=red.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import joblib
from pathlib import Path

R = Path('../results')
M = Path('../models')

# Poster style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.linestyle': ':',
    'grid.color': '#CCCCCC',
    'grid.alpha': 0.7,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Color palette: consistent across ALL figures
PALETTE = {
    'Popularity':  '#808080',
    'popularity':  '#808080',
    'CF':          '#2196F3',
    'cf':          '#2196F3',
    'Content':     '#4CAF50',
    'content':     '#4CAF50',
    'Hybrid':      '#F44336',
    'hybrid':      '#F44336',
    'CFColdStart': '#808080',
    'Hybrid-Adaptive':  '#F44336',
    'Hybrid-Fixed0.5':  '#FF9800',
    'Hybrid-RRF':       '#9C27B0',
}

SAV_KW = dict(dpi=300, bbox_inches='tight')
print('Style and palette loaded.')

In [ ]:
# ── Figure 1: Three-Protocol Comparison ─────────────────────────────────────
warm   = pd.read_csv(R / 'final_metrics.csv')
wlt    = pd.read_csv(R / 'warm_longtail_metrics.csv')
cold   = pd.read_csv(R / 'cold_start_metrics.csv')

protocols = [
    ('Warm (standard)', warm,   ['cf','hybrid','popularity','content'],
     ['CF','Hybrid','Popularity','Content']),
    ('Warm (long-tail)', wlt,   ['cf','hybrid','popularity','content'],
     ['CF','Hybrid','Popularity','Content']),
    ('Cold-start (standard)', cold, ['Popularity','Hybrid','Content','CFColdStart'],
     ['Popularity','Hybrid','Content','CF-ColdStart']),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, (title, df, models, labels) in zip(axes, protocols):
    means, los, his = [], [], []
    colors = []
    valid_labels = []
    for m, lbl in zip(models, labels):
        row = df[(df['model']==m)&(df['k']==10)&(df['metric']=='ndcg')]
        if row.empty:
            continue
        r = row.iloc[0]
        means.append(r['mean']); los.append(r['mean']-r['ci_low']); his.append(r['ci_high']-r['mean'])
        colors.append(PALETTE.get(m, '#888'))
        valid_labels.append(lbl)
    x = np.arange(len(means))
    ax.bar(x, means, color=colors, edgecolor='white', width=0.6,
           yerr=[los, his], capsize=5, error_kw=dict(elinewidth=1.5, capthick=1.5))
    ax.set_xticks(x); ax.set_xticklabels(valid_labels, rotation=15, ha='right')
    ax.set_ylabel('NDCG@10' if ax == axes[0] else '')
    ax.set_title(title)
    ax.yaxis.grid(True, linestyle=':', alpha=0.7); ax.xaxis.grid(False)

fig.suptitle('Model Performance Across Three Evaluation Protocols', fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(R / 'poster_three_protocol_comparison.png', **SAV_KW)
plt.close(fig)
print('Saved poster_three_protocol_comparison.png')

In [ ]:
# ── Figure 2: Cold-start Standard vs Long-tail ───────────────────────────────
cs_std = pd.read_csv(R / 'cold_start_metrics.csv')
cs_lt  = pd.read_csv(R / 'cold_start_metrics_longtail.csv')

models = ['Popularity', 'Hybrid', 'Content']
labels = ['Popularity', 'Hybrid', 'Content']

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, (df, title) in zip(axes, [(cs_std,'Cold-start Standard'), (cs_lt,'Cold-start Long-tail')]):
    means, los, his, colors = [], [], [], []
    valid = []
    for m, lbl in zip(models, labels):
        row = df[(df['model']==m)&(df['k']==10)&(df['metric']=='ndcg')]
        if row.empty: continue
        r = row.iloc[0]
        means.append(r['mean']); los.append(r['mean']-r['ci_low']); his.append(r['ci_high']-r['mean'])
        colors.append(PALETTE.get(m, '#888')); valid.append(lbl)
    x = np.arange(len(means))
    ax.bar(x, means, color=colors, edgecolor='white', width=0.5,
           yerr=[los, his], capsize=5, error_kw=dict(elinewidth=1.5, capthick=1.5))
    ax.set_xticks(x); ax.set_xticklabels(valid)
    ax.set_ylabel('NDCG@10' if ax == axes[0] else '')
    ax.set_title(title)
    ax.yaxis.grid(True, linestyle=':', alpha=0.7); ax.xaxis.grid(False)

axes[1].annotate('Popularity → 0\n(popularity bias)', xy=(0, 0.001),
                 xytext=(0.5, 0.008), fontsize=12, color='#808080',
                 arrowprops=dict(arrowstyle='->', color='#808080'))

fig.suptitle('Cold-Start: Standard vs Long-tail Evaluation (NDCG@10)', fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(R / 'poster_coldstart_standard_vs_longtail.png', **SAV_KW)
plt.close(fig)
print('Saved poster_coldstart_standard_vs_longtail.png')

In [ ]:
# ── Figure 3: Alpha Distribution ─────────────────────────────────────────────
user_item   = joblib.load(M / 'user_item_matrix.pkl')
user_to_idx = joblib.load(M / 'user_to_idx.pkl')
split       = joblib.load(R / 'eval_split.pkl')
sampled_users = split['sampled_users']

def compute_alpha(n):
    return 1.0 / (1.0 + np.exp(-((np.log1p(n) - 2.0) / 1.5)))

n_vals  = np.array([user_item[user_to_idx[u]].nnz for u in sampled_users])
alphas  = compute_alpha(n_vals)

# Sigmoid curve
n_range = np.linspace(0, max(n_vals)*1.05, 500)
curve   = compute_alpha(n_range)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(n_range, curve, color='#2196F3', lw=2.5, label='α(n) sigmoid')
ax1.axhline(0.5, color='gray', ls='--', lw=1.2, alpha=0.6)
ax1.set_xlabel('Number of training interactions (n_history)')
ax1.set_ylabel('Alpha (CF weight)', color='#2196F3')
ax1.tick_params(axis='y', labelcolor='#2196F3')
ax1.set_ylim(0, 1)

ax2.hist(n_vals, bins=40, color='#F44336', alpha=0.4, label='Test users')
ax2.set_ylabel('Number of test users', color='#F44336')
ax2.tick_params(axis='y', labelcolor='#F44336')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='center right', fontsize=12)

ax1.set_title(
    f'Adaptive Alpha Distribution — Test Users\n'
    f'α range: {alphas.min():.3f}–{alphas.max():.3f}  |  median {np.median(alphas):.3f}',
    fontsize=18,
)
ax1.grid(True, linestyle=':', alpha=0.5); ax1.set_axisbelow(True)
fig.tight_layout()
fig.savefig(R / 'poster_alpha_distribution.png', **SAV_KW)
plt.close(fig)
print('Saved poster_alpha_distribution.png')

In [ ]:
# ── Figure 4: Diversity Comparison ───────────────────────────────────────────
div = pd.read_csv(R / 'diversity_metrics.csv')
models_div = ['popularity','cf','content','hybrid']
labels_div = ['Popularity','CF','Content','Hybrid']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_div = [
    ('ils_mean', 'ils_ci_low', 'ils_ci_high', 'Intra-List Similarity (ILS)\n(lower = more diverse)', False),
    ('genre_cov_mean', 'genre_cov_low', 'genre_cov_high', 'Genre Coverage\n(fraction of k=10 with distinct genre)', False),
    ('pct_in_cf_cat', None, None, 'Catalog Coverage (% recs in CF catalog)', False),
]

for ax, (col, lo_col, hi_col, title, _) in zip(axes, metrics_div):
    means, los, his, colors, valid = [], [], [], [], []
    for m, lbl in zip(models_div, labels_div):
        row = div[div['model']==m]
        if row.empty or pd.isna(row[col].values[0]):
            means.append(0); los.append(0); his.append(0)
            colors.append(PALETTE.get(m,'#888')); valid.append(lbl + '\n(N/A)')
            continue
        r = row.iloc[0]
        means.append(float(r[col]))
        if lo_col and not pd.isna(r.get(lo_col, np.nan)):
            los.append(float(r[col]) - float(r[lo_col]))
            his.append(float(r[hi_col]) - float(r[col]))
        else:
            los.append(0); his.append(0)
        colors.append(PALETTE.get(m,'#888')); valid.append(lbl)
    x = np.arange(len(means))
    bars = ax.bar(x, means, color=colors, edgecolor='white', width=0.5)
    if any(l>0 or h>0 for l,h in zip(los,his)):
        ax.errorbar(x, means, yerr=[los,his], fmt='none', capsize=5,
                    elinewidth=1.5, capthick=1.5, color='#333')
    ax.set_xticks(x); ax.set_xticklabels(valid, rotation=10, ha='right')
    ax.set_title(title, fontsize=14)
    ax.yaxis.grid(True, linestyle=':', alpha=0.7); ax.xaxis.grid(False)

fig.suptitle('Diversity Metrics Across Models (k=10)', fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(R / 'poster_diversity_comparison.png', **SAV_KW)
plt.close(fig)
print('Saved poster_diversity_comparison.png')

In [ ]:
# ── Figure 5: Warm Hybrid Ablation ───────────────────────────────────────────
abl = pd.read_csv(R / 'warm_hybrid_ablation.csv')

ORDER  = ['CF','Hybrid-Adaptive','Hybrid-Fixed0.5','Hybrid-RRF']
XLBLS  = ['CF', 'Hybrid\nAdaptive', 'Hybrid\nFixed α=0.5', 'Hybrid\nRRF']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, protocol in zip(axes, ['standard','long-tail']):
    sub = abl[(abl['protocol']==protocol)&(abl['k']==10)&(abl['metric']=='ndcg')]
    means, los, his, colors = [], [], [], []
    for m in ORDER:
        row = sub[sub['model']==m]
        if row.empty: means.append(0); los.append(0); his.append(0); colors.append('#888'); continue
        r = row.iloc[0]
        means.append(r['mean']); los.append(r['mean']-r['ci_low']); his.append(r['ci_high']-r['mean'])
        colors.append(PALETTE.get(m,'#888'))
    x = np.arange(len(ORDER))
    ax.bar(x, means, color=colors, edgecolor='white', width=0.55,
           yerr=[los,his], capsize=5, error_kw=dict(elinewidth=1.5, capthick=1.5))
    ax.set_xticks(x); ax.set_xticklabels(XLBLS)
    ax.set_ylabel('NDCG@10' if ax==axes[0] else '')
    ax.set_title(f'Warm Ablation — {protocol.capitalize()}')
    ax.yaxis.grid(True, linestyle=':', alpha=0.7); ax.xaxis.grid(False)

fig.suptitle('Warm Hybrid Ablation: Blending Strategy Comparison (NDCG@10)', fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(R / 'poster_warm_hybrid_ablation.png', **SAV_KW)
plt.close(fig)
print('Saved poster_warm_hybrid_ablation.png')

In [ ]:
import os
for f in sorted(Path('../results').glob('poster_*.png')):
    print(f'{f.name}: {os.path.getsize(f)/1024:.0f} KB')